# Face Filter App: Traditional Computer Vision

This notebook builds a face filter using **traditional** methods only

**Pipeline for each frame**

1. Capture a frame from the webcam (`cv2.VideoCapture`) or upload a photo
2. Convert to greyscale and equalise the histogram
3. Detect faces and eyes with a **Viola-Jones Haar cascade** (`cv2.CascadeClassifier`)
4. Apply the filter 
5. Display the result


**Requirements:** OpenCV Contrib 4.x, NumPy and Matplotlib. Run this notebook locally so it can reach your webcam. 
OpenCV Contrib is an extension of the OpenCV (Open Source Computer Vision) library in Python. It extends the capabilities of the OpenCV library in Python by offering additional features and modules. These extras aren't in the main OpenCV library for reasons like being experimental, serving niche needs, or relying on third-party tools. With these added modules, you can do a lot more in computer vision, like detecting features, tweaking images, tracking objects, diving into machine learning, and much more.


In [101]:
# Install if needed - OpenCV Contrib 4.x  this is a prerequisite for using Haar cascades
# !pip install "opencv-contrib-python<5" numpy matplotlib

In [102]:
# Shows where python is running from
#import sys
#print(sys.executable)   # the Python the kernel is actually running
#print(sys.version)      # its Python version

In [103]:
# Show a list of installed Python libraries
# !pip list
# show the library details of opencv-contrib-python
# !pip show opencv-contrib-python

# Face Filter Application



## Imports

In [104]:
import tkinter as tk
from tkinter import ttk, filedialog
from PIL import Image, ImageTk
from tkinter import messagebox
import os
import cv2

###
# Importing face cascade
###
def import_cascades():
    CASCADE_DIR = "haarcascades/"   # local subfolder
    if not os.path.isdir(CASCADE_DIR):
        CASCADE_DIR = cv2.data.haarcascades

    face_cascade = cv2.CascadeClassifier(CASCADE_DIR+"haarcascade_frontalface_default.xml")
    eye_cascade = cv2.CascadeClassifier(CASCADE_DIR+"haarcascade_eye.xml")

    print("Face cascade loaded:", not face_cascade.empty())
    print("Eye cascade loaded: ", not eye_cascade.empty())

    return face_cascade, eye_cascade

## File Handling

### Video

In [105]:
# Updates video as its playing
def update_video():

    global current_frame, tracking, frame_count

    # adds 1 to frame count
    frame_count += 1

    # Checks if there is an active capture
    if cap is None:
        return

    # Reads in the next frame
    ret, frame = cap.read()

    # If there is no next frame, ends function
    if not ret:
        cap.release()
        return

    # Detects faces
    if not tracking:
        faces = detect_faces(frame)

        if len(faces) > 0:
            initialise_camshift(frame)
    else:
        faces = camshift_track(frame)

        # Refresh tracker every 10 frames
        if frame_count % 10 == 0:
            detected_faces = detect_faces(frame)

            if len(detected_faces) > 0:
                initialise_camshift(frame)


    # Creates copy of current frame to be used in webcam functions
    current_frame = frame.copy()
    # Gets current filter  from drop box
    current_filter = filter_option.get()
    # Applies filter to frame
    frame = apply_filter(current_filter, frame, faces)
    # Checks if webcam is recording, and there is an active writer
    if recording and webcam_writer is not None:
        # Writes the recorded frames to the writer
        webcam_writer.write(frame)

    # Convert to RGB to display in tkinter
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    # Takes image from array form (cv2) into tkinter
    img = Image.fromarray(frame_rgb)

    # Specifies what kind of image
    imgtk = ImageTk.PhotoImage(image=img)

    # Displays the video to the preview box
    display_label.imgtk = imgtk
    display_label.config(image=imgtk)

    root.after(15, update_video)

def open_video():

    global cap, current_video_path

    if cap is not None:
        cap.release()
    # Opens file explorer for user to select video with certain extensions
    path = filedialog.askopenfilename(
        filetypes=[
            ("Video Files", "*.mp4 *.avi *.webm *.mov"),
        ]
    )

    if not path:
        return

    current_video_path = path
    # Loads the video into cv2
    cap = cv2.VideoCapture(path)

    update_video()


### Image

In [106]:
def update_image():

    if current_image is None:
        return

    image = current_image.copy()

    faces = detect_faces(image)

    current_filter = filter_option.get()

    image = apply_filter(current_filter, image, faces)

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    img = Image.fromarray(image_rgb)
    img.thumbnail((900, 600))

    imgtk = ImageTk.PhotoImage(img)

    display_label.imgtk = imgtk
    display_label.config(image=imgtk)


def open_image():

    global current_image
    # Opens file explorer
    path = filedialog.askopenfilename(
        filetypes=[
            ("Image Files", "*.jpg *.png *.jpeg *.webm *.bmp"),
        ]
    )

    if not path:
        return

    current_image = cv2.imread(path)

    update_image()

### Webcam

In [107]:
def start_webcam():
    global cap

    if cap is not None:
        cap.release()
    # Captures native webcam of users device on a preview
    cap = cv2.VideoCapture(0)

    update_video()

#### Capture Photo

In [108]:
def capture_photo():
    # Sends error if webcam is not active
    if current_frame is None:
        messagebox.showwarning(
            "Capture",
            "No webcam frame"
        )
        return
    # Asks user where to save capture
    save_path = filedialog.asksaveasfilename(
        defaultextension=".jpg",
        filetypes=[
            ("JPEG", "*.jpg"),
            ("PNG", "*.png")
        ]
    )

    if not save_path:
        return

    image = current_frame.copy()

    filter_choice = filter_option.get()
    faces = detect_faces(image)

    image_filtered = apply_filter(filter_choice, image, faces)
    # Saves capture to specified path
    cv2.imwrite(save_path, image_filtered)

    messagebox.showinfo(
        "saved",
        "Photo saved successfully"
    )


#### Webcam Recording

In [109]:
def start_recording():

    global recording, webcam_writer

    if current_frame is None:
        messagebox.showwarning(
            "Capture",
            "No webcam frame"
        )
        return
    # Save path
    save_path = filedialog.asksaveasfilename(
        defaultextension=".mp4",
        filetypes=[
            ("MP4", "*.mp4")
        ]
    )

    if not save_path:
        return
    # Gets the height and width of the webcam frames
    height, width = current_frame.shape[:2]
    # Creates a writer at 30 fps
    webcam_writer = cv2.VideoWriter(
        save_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        30,
        (width, height)
    )
    recording = True

    messagebox.showinfo(
        "Recording",
        "Recording has started."
    )

def stop_recording():

    global recording, webcam_writer

    recording = False
    # If webcam writer is still active it ends
    if webcam_writer is not None:
        webcam_writer.release()
        webcam_writer = None

    messagebox.showinfo(
        "Recording",
        "Recording has finished."
    )


### Stop displaying

In [110]:
# Stops displaying any media in the preview box
def stop_video():

    global cap

    if cap:
        cap.release()
        cap = None

    display_label.config(
        image="",
        text="No media loaded"
    )

    display_label.imgtk = None


### Save Media

In [111]:
def save_media():

    if current_image is not None:
        save_image()

    elif current_video_path is not None:
        save_video()



def save_image():

    if current_image is None:
        return
    # Takes a copy of the current image displayed
    image = current_image.copy()
    # Adds the current  filter
    faces = detect_faces(image)
    current_filter = filter_option.get()

    image = apply_filter(current_filter, image, faces)
    # Asks for a save path
    save_path = filedialog.asksaveasfilename(
        defaultextension=".jpg",
        filetypes=[
            ("JPEG", "*.jpg"),
            ("PNG", "*.png")
        ]
    )
    # Saves image to specified path
    if save_path:
        cv2.imwrite(save_path, image)
        print("Image saved:", save_path)

def save_video():

    if current_video_path is None:
        return

    save_path = filedialog.asksaveasfilename(
        defaultextension=".mp4",
        filetypes=[
            ("MP4", "*.mp4")
        ]
    )

    if not save_path:
        return
    # Loads video to cv2
    video = cv2.VideoCapture(current_video_path)
    # Creates a video writer for the filtered video
    writer = cv2.VideoWriter(
        save_path,
        cv2.VideoWriter_fourcc(*"mp4v"),
        video.get(cv2.CAP_PROP_FPS),
        (int(video.get(cv2.CAP_PROP_FRAME_WIDTH)),
         int(video.get(cv2.CAP_PROP_FRAME_HEIGHT)))
    )

    while True:

        ret, frame = video.read()

        if not ret:
            break
        # Detects faces and applies filters
        faces = detect_faces(frame)

        frame = apply_filter(filter_option, frame, faces)

        writer.write(frame)

    video.release()
    writer.release()

    print("Video saved:", save_path)


## Detection & Tracking

### Face Detection


In [112]:
def detect_faces(frame):
    # sets the frame to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    #detects the coordinates of the faces
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=1.1,
        minNeighbors=5,
        minSize=(30, 30)
    )

    return faces

### Tracking

In [113]:
def initialise_camshift(frame):

    global track_window, roi_hist, tracking

    faces = detect_faces(frame)

    if len(faces) == 0:
        return

    (x, y, w, h) = faces[0]

    track_window = (x, y, w, h)

    roi = frame[y:y+h, x:x+w]

    hsv_roi = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

    mask = cv2.inRange(hsv_roi, (0, 60, 32), (180, 255, 255))

    roi_hist = cv2.calcHist([hsv_roi], [0], mask, [180], [0, 180])

    cv2.normalize(roi_hist, roi_hist, 0, 255, cv2.NORM_MINMAX)

    tracking = True

def camshift_track(frame):

     global track_window, roi_hist

     hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

     dst = cv2.calcBackProject([hsv],[0],roi_hist,[0,180],1)

     term_crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 1)

     ret, track_window = cv2.CamShift(dst, track_window, term_crit)
     print(track_window)
     x, y, w, h = track_window

     return [(x, y, w, h)]


## Filters

### Applying Filter

In [114]:
def apply_filter(filter_choice, frame, faces):
    #matches user choice to equivalent filter
    match filter_choice:
        case "None":
            return frame
        case "Blur Face":
            frame = blur_face(frame, faces)
        case "Gray Face":
            frame = grayscale_face(frame, faces)
        case "Edge Face":
            frame = edge_face(frame, faces)
        case _:
            print("Invalid filter: No filter Applied")

    return frame

def on_filter_change(event=None):
    # Updates image when a new filter is selected
    if current_image is not None:
        update_image()

### Filters

In [115]:
def blur_face(frame, faces):

    for (x, y, w, h) in faces:
       #gets region of interest of face
        roi = frame[y:y+h, x:x+w]

        #adds blur
        blurred_face = cv2.GaussianBlur(roi, (99, 99), 25)

        #merges roi with original frame
        frame[y:y+h, x:x+w] = blurred_face

    return frame

def grayscale_face(frame, faces):
    for (x, y, w, h) in faces:
       #gets region of interest of face
        roi = frame[y:y+h, x:x+w]

        #converts face to single channel grayscale
        grayscale_face = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        #converts back to 3 channels but it is still gray
        grayscale_face = cv2.cvtColor(grayscale_face, cv2.COLOR_GRAY2BGR)

        #merges roi with original frame
        frame[y:y+h, x:x+w] = grayscale_face

    return frame

def edge_face(frame, faces):
    for (x, y, w, h) in faces:

        roi = frame[y:y+h, x:x+w]

        gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)

        edges = cv2.Canny(gray, 100, 200)

        edges = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

        frame[y:y+h, x:x+w] = edges
    return frame

## Menu

### Filter Dropbox

In [116]:
def filter_choice_menu():
    global filter_option
    # Sets default value to no filter
    filter_option = tk.StringVar(value="None")

    # Drop down box that selects filter
    filter_menu = ttk.Combobox(
        root,
        textvariable=filter_option,
        values=[
            "None",
            "Blur Face",
            "Gray Face",
            "Edge Face"
        ],
        # Can only read and select the filters
        state="readonly"
    )
    filter_menu.pack()
    # Binds so image updates when filter is changed/selected
    filter_menu.bind("<<ComboboxSelected>>", on_filter_change
    )

### Window Setup

In [117]:
# Global Variables
cap = None
filter_option = None
current_image = None
current_video_path = None
current_frame = None
recording = False
webcam_writer = None
track_window = None
roi_hist = None
tracking = False
frame_count = 0
face_cascade, eye_cascade = import_cascades()

# Sets up the main display window
root = tk.Tk()
root.title("Face Filter System")
root.geometry("1100x900")

# Title
title_label = tk.Label(
    root,
    text="Face Filter System",
    font=("Arial", 20, "bold")
)
title_label.pack(pady=10)

# Filter Selection
filter_frame = tk.Frame(root)
filter_frame.pack(pady=5)

filter_label = tk.Label(
    filter_frame,
    text="Filter:",
)
filter_label.pack(side="left", padx=5)

filter_choice_menu()

# Toolbar for buttons
toolbar = tk.Frame(root)
toolbar.pack(pady=10)

# Display Area
display_frame = tk.Frame(
    root,
    width=900,
    height=600,
    relief="solid",
    borderwidth=2,
)

display_frame.pack(pady=20)
display_frame.pack_propagate(False)

display_label = tk.Label(
    display_frame,
    text="No media loaded",
    font=("Arial", 14)
)

display_label.pack(expand=True)

# Toolbar below for webcam
toolbarwebcam = tk.Frame(root)
toolbarwebcam.pack(pady=10)

# Buttons
video_button = tk.Button(
        toolbar,
        text="Open Video",
        width=15,
        command=open_video
    )

image_button = tk.Button(
        toolbar,
        text="Open Image",
        width=15,
        command=open_image
    )

stop_button = tk.Button(
        toolbar,
        text="Stop",
        width=15,
        command=stop_video
    )

save_button = tk.Button(
    toolbar,
    text="Save",
    width=15,
    command=save_media
)

video_button.pack(side="left", padx=5)
image_button.pack(side="left", padx=5)
save_button.pack(side="left", padx=5)
stop_button.pack(side="left", padx=5)

webcam_button = tk.Button(
        toolbarwebcam,
        text="Start Webcam",
        width="15",
        command=start_webcam
    )
capture_button = tk.Button(
    toolbarwebcam,
    text="Capture Webcam",
    width=15,
    command=capture_photo
)
start_recording_button = tk.Button(
    toolbarwebcam,
    text="Start Recording",
    width=15,
    command=start_recording
)
stop_recording_button = tk.Button(
    toolbarwebcam,
    text="Stop Recording",
    width=15,
    command=stop_recording
)
webcam_button.pack(side="left", padx=5)
capture_button.pack(side="left", padx=5)
start_recording_button.pack(side="left", padx=5)
stop_recording_button.pack(side="left", padx=5)


# Loads tk window
root.mainloop()


Face cascade loaded: True
Eye cascade loaded:  True
(271, 72, 138, 222)
(290, 76, 129, 286)
(279, 218, 157, 262)
(263, 242, 188, 238)
(251, 264, 210, 216)
(239, 278, 235, 202)
(211, 286, 292, 194)
(176, 292, 364, 188)
(162, 300, 431, 180)
(276, 71, 133, 216)
(299, 80, 119, 279)
(292, 218, 144, 262)
(281, 238, 169, 242)
(273, 258, 197, 222)
(289, 272, 221, 208)
(291, 282, 268, 198)
(274, 294, 311, 186)
(243, 302, 356, 178)
(163, 304, 432, 176)
(279, 69, 133, 226)
(296, 82, 128, 298)
(287, 214, 152, 266)
(276, 240, 179, 240)
(266, 264, 208, 216)
(270, 278, 253, 202)
(274, 290, 302, 190)
(241, 300, 351, 180)
(168, 302, 424, 178)
(133, 308, 475, 172)
(288, 74, 131, 220)
(304, 74, 119, 275)
(300, 108, 135, 345)
(292, 150, 151, 330)
(285, 170, 167, 310)
(276, 198, 186, 282)
(263, 224, 214, 256)
(264, 252, 251, 228)
(269, 276, 293, 204)
(250, 292, 339, 188)
(299, 77, 126, 211)
(310, 74, 114, 248)
(315, 73, 117, 297)
(312, 69, 126, 348)
(308, 63, 135, 387)
(304, 68, 144, 412)
(303, 84, 151, 39